In [ ]:
import random
import warnings

import numpy as np
import pandas as pd

from sktime.forecasting.arima import ARIMA, AutoARIMA
from sktime.utils.plotting import plot_correlations

import helpers as h

%load_ext autoreload
%autoreload 2

In [ ]:
# Default image properties
properties = {
    "width": 1000,
    "height": 250,
}

# Filter sktime deprecation warnings.
warnings.filterwarnings(action="ignore", category=FutureWarning)

## ARIMA(1, 0, 0): Only auto-correlation

In [ ]:
# Set random seeds
seed = 4
random.seed(seed)
np.random.seed(seed)

In [ ]:
# Generate AR(1) process data.
n = 50
coef = 0.9

# Generate normally distributed noise.
y = np.random.normal(0, 1, size=n)

# Generate auto-correlation using specified coefficient.
for i in range(n):
    if i > 0:
        y[i] += coef * y[i - 1]


ar1 = pd.Series(y)

In [ ]:
h.plot_timeseries(ar1, properties=properties).configure(padding=30)

In [ ]:
forecaster = ARIMA(order=(1, 0, 0), with_intercept=False)
forecaster.fit(ar1)

In [ ]:
forecaster.summary()

In [ ]:
# Get 90% prediction interval.
forecaster.predict_interval(fh=[1, 2, 3, 4, 5], coverage=0.9)

In [ ]:
# Same using quantiles.
forecaster.predict_quantiles(fh=[1, 2, 3, 4, 5], alpha=[0.05, 0.95])

In [ ]:
# Plot forecaster with forecast
h.plot_forecaster(
    ar1,
    forecaster,
    plot_horizon=20,
    colors={"values": "#1f77b4", "forecast": "#ff7f0e"},
    properties=properties,
).configure(padding=30)

In [ ]:
# plot_correlations(ar1)

## ARIMA([7], 1, 0): Seasonality and trend

In [ ]:
# Generate a time series with a linear trend and seasonality.
n = 50
intercept = 10
slope = 2
seasonality = [29, 30, 25, 27, 31, 43, 37]

ar7 = h.generate_linear(n, slope=slope, intercept=intercept)
ar7 = h.add_seasonality(ar7, seasonality, 3)
ar7 = h.add_noise(ar7, sd=3)

h.plot_timeseries(ar7, properties=properties).configure(padding=30)

In [ ]:
forecaster = ARIMA(order=([7], 1, 0), with_intercept=False)
forecaster.fit(ar7)

In [ ]:
forecaster.summary()

In [ ]:
# Plot forecaster with forecast
h.plot_forecaster(
    ar7,
    forecaster,
    plot_horizon=20,
    colors={"values": "#1f77b4", "forecast": "#ff7f0e"},
    properties=properties,
).configure(padding=30)

In [ ]:
# plot_correlations(ar7.diff(1).dropna(), lags=20)

## ARIMA(0, 0, 1): Moving average only

In [ ]:
# Generate MA(1) process data.
n = 100
intercept = 20
coef = 0.8


noise = pd.Series(np.random.normal(0, 1, n))
ma1 = intercept + noise + coef * noise.shift(1).fillna(0)

ma1.plot(marker=".")

In [ ]:
forecaster = ARIMA(order=(0, 0, 1), with_intercept=True)
forecaster.fit(ma1)

In [ ]:
forecaster.summary()

In [ ]:
h.plot_forecaster(ma1, forecaster, plot_horizon=20, colors={"values": "#1f77b4", "forecast": "#ff7f0e"}, properties=properties)

In [ ]:
# plot_correlations(ma1)

## Plot correlations

In [ ]:
# Check autocorrelation for stationairy series.
np.random.seed(2)
plot_correlations(
    h.generate_autoregression(n=150, coef=0.9),
    lags=20
)
None

In [ ]:
# Autocorrelation for AR with trend.
np.random.seed(2)
n = 150

plot_correlations(
    h.generate_linear(n, slope=.15) + h.generate_autoregression(n=150, coef=0.9),
    lags=20
)
None

In [ ]:
# Generate a time series with a linear trend and seasonality.
np.random.seed(2)

seasonality = [29, 30, 25, 27, 31, 43, 37]
raw = pd.Series([0] * 150)
raw = h.add_seasonality(raw, seasonality, 1)
raw = h.add_noise(raw, sd=1)

plot_correlations(raw, lags=20)
None

In [ ]:
forecaster = ARIMA(order=([1, 7], 0, 0), with_intercept=False)
forecaster.fit(raw)
forecaster.summary()

In [ ]:
h.plot_forecaster(
    raw,
    forecaster,
    plot_horizon=20,
    colors={"values": "#1f77b4", "forecast": "#ff7f0e"},
    properties=properties,
).configure(padding=30)